### Created By: ESAB
### Gravity BookStore DWH
### Date: 13-04-2026

# ETL Using Python
## 1 - Extract

In [1]:
!pip install pyodbc


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pyodbc 
import pandas as pd

In [4]:
pyodbc.drivers()  # List available ODBC drivers

['SQL Server',
 'ODBC Driver 17 for SQL Server',
 'ODBC Driver 18 for SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']

In [101]:
# Establish connection to the source database
source_conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=.\SQLEXPRESS;"
    "DATABASE=BookStore_EG;"
    "Trusted_Connection=yes;"
    )

In [13]:
# CustomerDim table has: CustomerID, FullName, City
customer_df = pd.read_sql("select * from customer_view;", source_conn)
customer_df

C:\Users\5g\AppData\Local\Temp\ipykernel_10628\3780264122.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_df = pd.read_sql("select * from customer_view;", source_conn)


,CustomerID,FullName,City
0,1,Shaker,None
1,2,El-Gamal,None
2,3,Asmaa,Tanta
3,4,Rania,Giza
4,5,Nour Ibrahim,None
...,...,...,...
1995,1996,Sara Hassan,None
1996,1997,,None
1997,1998,El-Naggar,None
1998,1999,,None


In [25]:
# BookDim table has: BookID, Title,CategoryDescription, AuthorName, Price, BookAge
book_df = pd.read_sql("select * from book_view;",source_conn)
book_df

C:\Users\5g\AppData\Local\Temp\ipykernel_10628\3735496179.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  book_df = pd.read_sql("select * from book_view;",source_conn)


,BookID,Title,CategoryDescription,AuthorName,Price,BookAge
0,1,Chemistry Today 1,Chemistry,Fekry Abaza,72.66,25
1,1,Chemistry Today 1,Chemistry,Ahmed Abdel Moati Hegazy,72.66,25
2,1,Chemistry Today 1,Chemistry,Ahmed El-Desouky,72.66,25
3,2,Islamic Studies 2,Religion,Baligh Hamdy,80.34,22
4,2,Islamic Studies 2,Religion,Ahmed El-Hefnawy,80.34,22
...,...,...,...,...,...,...
3004,999,Political Analysis 999,Politics,Mohamed El-Darwish,66.50,5
3005,1000,Physics Explained 1000,Physics,Abdelrahman Mounif,133.67,12
3006,1000,Physics Explained 1000,Physics,Abdel Muti Hegazi,133.67,12
3007,1000,Physics Explained 1000,Physics,Hassan Talaat,133.67,12


In [102]:
# FactOrder table has: OrderID, CustomerID, BookID, OrderDate, Quantity, UnitPrice, TotalAmount
factorders_df = pd.read_sql("select * from fact_view;", source_conn)
factorders_df

C:\Users\5g\AppData\Local\Temp\ipykernel_10628\1228968611.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  factorders_df = pd.read_sql("select * from fact_view;", source_conn)


,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice
0,1,NaN,198,2020-12-17,1.0,180.70
1,1,NaN,256,2020-12-17,2.0,114.14
2,1,NaN,393,2020-12-17,1.0,188.30
3,1,NaN,492,2020-12-17,1.0,218.49
4,1,NaN,996,2020-12-17,1.0,NaN
...,...,...,...,...,...,...
32029,7999,790.0,910,2021-05-11,2.0,142.16
32030,7999,790.0,951,2021-05-11,NaN,81.40
32031,8000,713.0,49,2022-03-15,2.0,237.53
32032,8000,713.0,217,2022-03-15,1.0,78.51


In [119]:
source_conn.close()

## 2 - Transform

In [18]:
customer_df.head()

,CustomerID,FullName,City
0,1,Shaker,None
1,2,El-Gamal,None
2,3,Asmaa,Tanta
3,4,Rania,Giza
4,5,Nour Ibrahim,None


In [19]:
customer_df.dtypes

CustomerID     int64
FullName      object
City          object
dtype: object

In [20]:
customer_df.isnull().sum()

CustomerID      0
FullName        0
City          872
dtype: int64

In [21]:
# Since there are 873 none in the City column, we can fill them with 'Unknown' or 'Not Provided'
customer_df['City'] = customer_df['City'].fillna('Unknown')
customer_df.isnull().sum()

CustomerID    0
FullName      0
City          0
dtype: int64

In [22]:
customer_df.head()

,CustomerID,FullName,City
0,1,Shaker,Unknown
1,2,El-Gamal,Unknown
2,3,Asmaa,Tanta
3,4,Rania,Giza
4,5,Nour Ibrahim,Unknown


In [27]:
book_df.head()

,BookID,Title,CategoryDescription,AuthorName,Price,BookAge
0,1,Chemistry Today 1,Chemistry,Fekry Abaza,72.66,25
1,1,Chemistry Today 1,Chemistry,Ahmed Abdel Moati Hegazy,72.66,25
2,1,Chemistry Today 1,Chemistry,Ahmed El-Desouky,72.66,25
3,2,Islamic Studies 2,Religion,Baligh Hamdy,80.34,22
4,2,Islamic Studies 2,Religion,Ahmed El-Hefnawy,80.34,22


In [29]:
book_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3009 entries, 0 to 3008
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   BookID               3009 non-null   int64  
 1   Title                3009 non-null   object 
 2   CategoryDescription  3009 non-null   object 
 3   AuthorName           2959 non-null   object 
 4   Price                3009 non-null   float64
 5   BookAge              3009 non-null   int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 141.2+ KB


In [35]:
# check nulls
book_df.isnull().sum()

BookID                  0
Title                   0
CategoryDescription     0
AuthorName             50
Price                   0
BookAge                 0
dtype: int64

In [36]:
# there are no null values in the author name, we can fill them with 'Unknown' or 'Not Provided'
book_df['AuthorName'] = book_df['AuthorName'].fillna('Unknown')

In [37]:
book_df.isnull().sum()

BookID                 0
Title                  0
CategoryDescription    0
AuthorName             0
Price                  0
BookAge                0
dtype: int64

In [38]:
#Change BookAge to float as defined in DWH
book_df['BookAge'] = book_df['BookAge'].astype(float)
book_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3009 entries, 0 to 3008
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   BookID               3009 non-null   int64  
 1   Title                3009 non-null   object 
 2   CategoryDescription  3009 non-null   object 
 3   AuthorName           3009 non-null   object 
 4   Price                3009 non-null   float64
 5   BookAge              3009 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 141.2+ KB


In [103]:
factorders_df.head()

,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice
0,1,NaN,198,2020-12-17,1.0,180.70
1,1,NaN,256,2020-12-17,2.0,114.14
2,1,NaN,393,2020-12-17,1.0,188.30
3,1,NaN,492,2020-12-17,1.0,218.49
4,1,NaN,996,2020-12-17,1.0,NaN


In [104]:
factorders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32034 entries, 0 to 32033
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   orderID     32034 non-null  int64  
 1   CustomerID  30661 non-null  float64
 2   BookID      32034 non-null  int64  
 3   OrderDate   32034 non-null  object 
 4   Quantity    30523 non-null  float64
 5   UnitPrice   30414 non-null  float64
dtypes: float64(3), int64(2), object(1)
memory usage: 1.5+ MB


In [105]:
factorders_df.isnull().sum()

orderID          0
CustomerID    1373
BookID           0
OrderDate        0
Quantity      1511
UnitPrice     1620
dtype: int64

In [106]:
#CustomerID has 1373 Null values, we can fill them with 0
factorders_df['CustomerID'] = factorders_df['CustomerID'].fillna(0)
#Quantity has 1511 Null values, we can fill them with 1
factorders_df['Quantity'] = factorders_df['Quantity'].fillna(1).astype(int)
#UnitPrice has 1620 Null values, we can fill them with default price from book_df by merging the two dataframes on BookID
#merging only if there are null values in UnitPrice column
factorders_df = factorders_df.merge(book_df[['BookID', 'Price']], on='BookID', how='left')
factorders_df['UnitPrice'] = factorders_df['UnitPrice'].fillna(factorders_df['Price'])
#drop the Price column after filling the null values in UnitPrice
factorders_df.drop('Price', axis=1, inplace=True)
factorders_df

,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice
0,1,0.0,198,2020-12-17,1,180.70
1,1,0.0,198,2020-12-17,1,180.70
2,1,0.0,256,2020-12-17,2,114.14
3,1,0.0,256,2020-12-17,2,114.14
4,1,0.0,256,2020-12-17,2,114.14
...,...,...,...,...,...,...
96566,7999,790.0,951,2021-05-11,1,81.40
96567,7999,790.0,951,2021-05-11,1,81.40
96568,8000,713.0,49,2022-03-15,2,237.53
96569,8000,713.0,217,2022-03-15,1,78.51


In [107]:
factorders_df.isnull().sum()

orderID       0
CustomerID    0
BookID        0
OrderDate     0
Quantity      0
UnitPrice     0
dtype: int64

In [108]:
#change CustomerID to int as defined in DWH
factorders_df['CustomerID'] = factorders_df['CustomerID'].astype(int)
factorders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96571 entries, 0 to 96570
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   orderID     96571 non-null  int64  
 1   CustomerID  96571 non-null  int64  
 2   BookID      96571 non-null  int64  
 3   OrderDate   96571 non-null  object 
 4   Quantity    96571 non-null  int64  
 5   UnitPrice   96571 non-null  float64
dtypes: float64(1), int64(4), object(1)
memory usage: 4.4+ MB


In [109]:
factorders_df.head()

,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice
0,1,0,198,2020-12-17,1,180.70
1,1,0,198,2020-12-17,1,180.70
2,1,0,256,2020-12-17,2,114.14
3,1,0,256,2020-12-17,2,114.14
4,1,0,256,2020-12-17,2,114.14


In [110]:
# convert OrderDate to datetime format
factorders_df['OrderDate'] = pd.to_datetime(factorders_df['OrderDate'], errors='coerce')
#convert it into int format as defined in DWH
factorders_df['OrderDate'] = factorders_df['OrderDate'].dt.strftime('%Y%m%d').astype(int)
factorders_df.head()

,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice
0,1,0,198,20201217,1,180.70
1,1,0,198,20201217,1,180.70
2,1,0,256,20201217,2,114.14
3,1,0,256,20201217,2,114.14
4,1,0,256,20201217,2,114.14


In [111]:
# Add column TotalAmount by multiplying Quantity and UnitPrice
factorders_df['TotalAmount'] = factorders_df['Quantity'] * factorders_df['UnitPrice']
factorders_df.head()

,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice,TotalAmount
0,1,0,198,20201217,1,180.70,180.70
1,1,0,198,20201217,1,180.70,180.70
2,1,0,256,20201217,2,114.14,228.28
3,1,0,256,20201217,2,114.14,228.28
4,1,0,256,20201217,2,114.14,228.28


In [112]:
factorders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96571 entries, 0 to 96570
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   orderID      96571 non-null  int64  
 1   CustomerID   96571 non-null  int64  
 2   BookID       96571 non-null  int64  
 3   OrderDate    96571 non-null  int64  
 4   Quantity     96571 non-null  int64  
 5   UnitPrice    96571 non-null  float64
 6   TotalAmount  96571 non-null  float64
dtypes: float64(2), int64(5)
memory usage: 5.2 MB


## 3 - Load

In [120]:
# Establish connection to the Destination database TO DWH
destination_conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=.\SQLEXPRESS;"
    "DATABASE=Book_DWH;"
    "Trusted_Connection=yes;"
    )

In [76]:
for index, row in customer_df.iterrows():
    cursor = destination_conn.cursor()
    cursor.execute("INSERT INTO CustomerDim (CustomerID, FullName,City) VALUES (?, ?, ?)",
                   row['CustomerID'], row['FullName'], row['City'])
    destination_conn.commit()

In [81]:
for index, row in book_df.iterrows():
    cursor = destination_conn.cursor()
    cursor.execute("INSERT INTO BookDim (BookID, Title,CategoryDescription, AuthorName, Price, BookAge) VALUES (?, ?, ?, ?, ?, ?)",
                   row['BookID'], row['Title'], row['CategoryDescription'], row['AuthorName'], row['Price'], row['BookAge'])
    destination_conn.commit()

In [88]:
factorders_df.head()

,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice,TotalAmount
0,1,0,198,20201217,1,180.70,180.70
1,1,0,198,20201217,1,180.70,180.70
2,1,0,198,20201217,1,180.70,180.70
3,1,0,198,20201217,1,180.70,180.70
4,1,0,256,20201217,2,114.14,228.28


In [98]:
book_dim = pd.read_sql("select * from BookDim;", destination_conn)
book_dim.head()

C:\Users\5g\AppData\Local\Temp\ipykernel_10628\1804261873.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  book_dim = pd.read_sql("select * from BookDim;", destination_conn)


,Book_SK,BookID,Title,CategoryDescription,AuthorName,Price,BookAge
0,1,1,Chemistry Today 1,Chemistry,Fekry Abaza,72.66,25.0
1,2,1,Chemistry Today 1,Chemistry,Ahmed Abdel Moati Hegazy,72.66,25.0
2,3,1,Chemistry Today 1,Chemistry,Ahmed El-Desouky,72.66,25.0
3,4,2,Islamic Studies 2,Religion,Baligh Hamdy,80.34,22.0
4,5,2,Islamic Studies 2,Religion,Ahmed El-Hefnawy,80.34,22.0


In [114]:
factorders_df = pd.merge(factorders_df, book_dim, how='inner', on='BookID')
factorders_df.head()

,orderID,CustomerID,BookID,OrderDate,Quantity,UnitPrice,TotalAmount,Book_SK,Title,CategoryDescription,AuthorName,Price,BookAge
0,1,0,198,20201217,1,180.70,180.70,603,Engineering Principles 198,Engineering,Nael Eltoukhy,180.70,3.0
1,1,0,198,20201217,1,180.70,180.70,604,Engineering Principles 198,Engineering,Ahmed El-Desouky,180.70,3.0
2,1,0,198,20201217,1,180.70,180.70,603,Engineering Principles 198,Engineering,Nael Eltoukhy,180.70,3.0
3,1,0,198,20201217,1,180.70,180.70,604,Engineering Principles 198,Engineering,Ahmed El-Desouky,180.70,3.0
4,1,0,256,20201217,2,114.14,228.28,772,Islamic Studies 256,Religion,Ibrahim Aslan,114.14,5.0


In [ ]:
# Take only the columns needed for FactOrders table
factorders_df = factorders_df[['orderID', 'CustomerID', 'Book_SK', 'OrderDate', 'Quantity', 'UnitPrice', 'TotalAmount']]

In [122]:
for index, row in factorders_df.iterrows():
    cursor = destination_conn.cursor()
    cursor.execute("INSERT INTO FactOrders (OrderID, CustomerID, BookID, OrderDate, Quantity, UnitPrice, TotalAmount) VALUES (?, ?, ?, ?, ?, ?, ?)",
                   row['orderID'], row['CustomerID'], row['Book_SK'], row['OrderDate'], row['Quantity'], row['UnitPrice'], row['TotalAmount'])
    destination_conn.commit()

KeyboardInterrupt: 

In [88]:
cursor.close()
destination_conn.close()